# Anomaly Score Contribution Analysis

This notebook analyzes which sensor channels contribute most to the anomaly scores.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline


## 1. Load Data


In [ ]:
# Configuration
results_dir = Path('data/results/20251229_232628')
processed_dir = Path('data/processed')
panel_id = '1'  # Change to '2' for panel 2

# Load results
forecast_df = pd.read_csv(
    results_dir / f'LightGBM_panel_{panel_id}_forecast.csv',
    index_col='timestamp',
    parse_dates=True
)

scores_df = pd.read_csv(
    results_dir / f'LightGBM_panel_{panel_id}_scores.csv',
    index_col='timestamp',
    parse_dates=True
)

flags_df = pd.read_csv(
    results_dir / f'LightGBM_panel_{panel_id}_flags.csv',
    index_col='timestamp',
    parse_dates=True
)

# Load actual data
actual_df = pd.read_parquet(processed_dir / f'panel_{panel_id}.parquet')

print(f"Forecast shape: {forecast_df.shape}")
print(f"Scores shape: {scores_df.shape}")
print(f"Actual data shape: {actual_df.shape}")
print(f"\nForecast columns: {list(forecast_df.columns)}")


## 2. Calculate Residuals


In [ ]:
# Align actual data to forecast timestamps
actual_aligned = actual_df.loc[forecast_df.index]

# Calculate residuals (actual - predicted)
residuals_df = actual_aligned - forecast_df

# Calculate absolute residuals for contribution analysis
abs_residuals_df = residuals_df.abs()

# Calculate squared residuals (for variance-based contribution)
squared_residuals_df = residuals_df ** 2

print("Residuals calculated successfully!")
print(f"\nResiduals shape: {residuals_df.shape}")
print(f"\nResiduals statistics:")
print(residuals_df.describe())


## 3. Overall Feature Contribution Analysis


In [ ]:
# Calculate mean absolute residuals for each feature
mean_abs_residuals = abs_residuals_df.mean().sort_values(ascending=False)

# Calculate RMSE for each feature
rmse_per_feature = np.sqrt(squared_residuals_df.mean()).sort_values(ascending=False)

# Create figure with subplots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot mean absolute residuals
mean_abs_residuals.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_xlabel('Mean Absolute Residual')
axes[0].set_title('Average Prediction Error by Feature')
axes[0].grid(axis='x', alpha=0.3)

# Plot RMSE
rmse_per_feature.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_xlabel('RMSE')
axes[1].set_title('Root Mean Squared Error by Feature')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nTop 5 features by mean absolute residual:")
print(mean_abs_residuals.head())


## 4. High-Score Period Analysis


In [ ]:
# Define high-score threshold (e.g., 95th percentile)
high_score_threshold = scores_df['anomaly_score'].quantile(0.95)
high_score_mask = scores_df['anomaly_score'] > high_score_threshold

print(f"High score threshold (95th percentile): {high_score_threshold:.3f}")
print(f"Number of high-score timestamps: {high_score_mask.sum()}")

# Calculate feature contributions during high-score periods
high_score_contributions = abs_residuals_df[high_score_mask].mean().sort_values(ascending=False)
normal_score_contributions = abs_residuals_df[~high_score_mask].mean().sort_values(ascending=False)

# Calculate the difference (what makes high-score periods different)
contribution_diff = (high_score_contributions - normal_score_contributions).sort_values(ascending=False)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# High score period contributions
high_score_contributions.plot(kind='barh', ax=axes[0], color='crimson')
axes[0].set_xlabel('Mean Absolute Residual')
axes[0].set_title('Feature Contributions During High-Score Periods')
axes[0].grid(axis='x', alpha=0.3)

# Difference from normal periods
contribution_diff.plot(kind='barh', ax=axes[1], color='purple')
axes[1].set_xlabel('Residual Increase vs Normal Periods')
axes[1].set_title('What Makes High-Score Periods Different')
axes[1].grid(axis='x', alpha=0.3)
axes[1].axvline(x=0, color='black', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print("\nTop contributors during high-score periods:")
print(high_score_contributions.head())
print("\nFeatures with highest increase during high-score periods:")
print(contribution_diff.head())


## 5. Time Series Visualization


In [ ]:
# Plot anomaly scores over time with top contributing features
top_features = mean_abs_residuals.head(3).index

fig, axes = plt.subplots(len(top_features) + 1, 1, figsize=(14, 10), sharex=True)

# Plot anomaly scores
axes[0].plot(scores_df.index, scores_df['anomaly_score'], label='Anomaly Score', color='black', linewidth=1.5)
axes[0].axhline(y=high_score_threshold, color='red', linestyle='--', alpha=0.5, label='95th Percentile')
axes[0].fill_between(scores_df.index, 0, scores_df['anomaly_score'], 
                      where=high_score_mask, alpha=0.3, color='red', label='High Score Periods')
axes[0].set_ylabel('Anomaly Score')
axes[0].set_title('Anomaly Scores Over Time')
axes[0].legend(loc='upper right')
axes[0].grid(alpha=0.3)

# Plot top contributing features' absolute residuals
for i, feature in enumerate(top_features, 1):
    axes[i].plot(abs_residuals_df.index, abs_residuals_df[feature], label=feature, linewidth=1.5)
    axes[i].fill_between(abs_residuals_df.index, 0, abs_residuals_df[feature],
                         where=high_score_mask, alpha=0.3, color='red')
    axes[i].set_ylabel('Abs Residual')
    axes[i].set_title(f'{feature} - Absolute Residuals')
    axes[i].legend(loc='upper right')
    axes[i].grid(alpha=0.3)

axes[-1].set_xlabel('Timestamp')
plt.tight_layout()
plt.show()


## 6. Heatmap of Residuals


In [ ]:
# Create a heatmap of normalized absolute residuals
# Normalize each feature to [0, 1] for better comparison
normalized_residuals = abs_residuals_df.copy()
for col in normalized_residuals.columns:
    max_val = normalized_residuals[col].max()
    if max_val > 0:
        normalized_residuals[col] = normalized_residuals[col] / max_val

# Create heatmap
fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(normalized_residuals.T, cmap='YlOrRd', cbar_kws={'label': 'Normalized Abs Residual'},
            xticklabels=20, yticklabels=True, ax=ax)
ax.set_xlabel('Timestamp Index')
ax.set_ylabel('Feature')
ax.set_title('Heatmap of Normalized Absolute Residuals Over Time')
plt.tight_layout()
plt.show()


## 7. Individual High-Score Instance Analysis


In [ ]:
# Get top 10 highest anomaly scores
top_anomalies = scores_df.nlargest(10, 'anomaly_score')

print("Top 10 Anomaly Scores:")
print("=" * 80)

for idx, (timestamp, row) in enumerate(top_anomalies.iterrows(), 1):
    score = row['anomaly_score']
    print(f"\n{idx}. Timestamp: {timestamp}")
    print(f"   Anomaly Score: {score:.4f}")
    print(f"   Is Flagged: {'Yes' if flags_df.loc[timestamp, 'anomaly_flag'] == 1 else 'No'}")
    print(f"\n   Top 5 Contributing Features:")
    
    feature_residuals = abs_residuals_df.loc[timestamp].sort_values(ascending=False)
    for feature, residual in feature_residuals.head(5).items():
        actual_val = actual_aligned.loc[timestamp, feature]
        pred_val = forecast_df.loc[timestamp, feature]
        print(f"      - {feature:30s}: residual={residual:8.4f} (actual={actual_val:8.4f}, pred={pred_val:8.4f})")
    print("-" * 80)


## 8. Correlation Between Features and Anomaly Score


In [ ]:
# Calculate correlation between absolute residuals and anomaly scores
correlations = abs_residuals_df.corrwith(scores_df['anomaly_score']).sort_values(ascending=False)

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
correlations.plot(kind='barh', ax=ax, color='teal')
ax.set_xlabel('Correlation with Anomaly Score')
ax.set_title('Feature Residual Correlation with Overall Anomaly Score')
ax.axvline(x=0, color='black', linestyle='-', alpha=0.3)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("Features most correlated with anomaly scores:")
print(correlations.head())


## 9. Approximate Per-Feature Score Reconstruction

Since the KMeans scorer uses L2 distance on the scaled residual vector, we can approximate how much each feature contributes to the overall score.


In [ ]:
# Approximate contribution: squared residuals normalized by total squared error
total_squared_residuals = squared_residuals_df.sum(axis=1)
feature_contributions = squared_residuals_df.div(total_squared_residuals, axis=0)

# Weight by anomaly score to get approximate score contributions
feature_score_contributions = feature_contributions.mul(scores_df['anomaly_score'], axis=0)

# Average contribution per feature
avg_score_contribution = feature_score_contributions.mean().sort_values(ascending=False)

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
avg_score_contribution.plot(kind='barh', ax=ax, color='darkgreen')
ax.set_xlabel('Average Score Contribution')
ax.set_title('Estimated Average Contribution to Anomaly Score by Feature')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\nEstimated average score contributions:")
print(avg_score_contribution)


## 10. Export Results


In [ ]:
# Create output directory
output_dir = results_dir / 'analysis'
output_dir.mkdir(exist_ok=True)

# Save residuals
residuals_df.to_csv(output_dir / f'panel_{panel_id}_residuals.csv')
abs_residuals_df.to_csv(output_dir / f'panel_{panel_id}_abs_residuals.csv')

# Save summary statistics
summary_stats = pd.DataFrame({
    'mean_abs_residual': mean_abs_residuals,
    'rmse': rmse_per_feature,
    'correlation_with_score': correlations,
    'avg_score_contribution': avg_score_contribution,
    'high_score_contribution': high_score_contributions,
    'normal_score_contribution': normal_score_contributions,
    'contribution_diff': contribution_diff
})
summary_stats.to_csv(output_dir / f'panel_{panel_id}_feature_summary.csv')

# Save per-timestamp feature contributions
feature_score_contributions.to_csv(output_dir / f'panel_{panel_id}_feature_score_contributions.csv')

print(f"\nAnalysis results saved to: {output_dir}")
print(f"\nFiles created:")
for file in output_dir.glob('*.csv'):
    print(f"  - {file.name}")


## Summary

This notebook provides multiple perspectives on what's driving your anomaly scores:

1. **Overall Feature Contribution**: Which features have the largest prediction errors on average
2. **High-Score Period Analysis**: Which features contribute most during high-anomaly periods
3. **Time Series**: How individual features behave over time relative to anomaly scores
4. **Correlation Analysis**: Which features are most correlated with high anomaly scores
5. **Score Reconstruction**: Approximate contribution of each feature to the overall anomaly score

Use these insights to:
- Understand which sensors are most problematic
- Identify if certain features need better forecasting models
- Determine if some features should be weighted differently
- Guide feature engineering or data quality improvements
